# OrgaLego - A Tool for Organizational Restructuring
For the long output cells please view them as a scrollable element



# Part 1 - Setup, loading, overview

In [24]:
from orgalego.core.loader import load_graph

master = load_graph("./data/org_master.json")
print(f'{len(master)} nodes, {len(master.edges)} edges')
print('Validation:', 'OK' if not master.validate() else master.validate())

700 nodes, 1376 edges
Validation: OK


#### Researche by name : 

In [25]:
def search_node(graph, keyword):
    keyword = keyword.lower()
    for node_id, node in graph.nodes.items():
        if keyword in node.label.lower() or keyword in node.role_family.lower():
            print(f'{node_id} | {node.label} | {node.role_family} | rank={node.rank}')

search_node(master, 'Lambert')

79 | Lynlee Lambert | support_staff | rank=2.0
96 | Annaleigha Lambert | coordinator | rank=4.0
101 | Blayklee Lambert | student | rank=3.5
178 | Ebraheem Lambert | teacher | rank=3.5
231 | Sharvin Lambert | engineer | rank=2.0
250 | Alaina Lambert | student | rank=4.0
285 | Yugan Lambert | coordinator | rank=2.0
312 | Khamden Lambert | data_analyst | rank=3.0
379 | Ayodeji Lambert | researcher | rank=3.0
398 | Ellanor Lambert | support_staff | rank=4.0
408 | Durham Lambert | support_staff | rank=4.0
410 | Amyracle Lambert | coordinator | rank=2.5
463 | Jannel Lambert | researcher | rank=4.0
471 | Arinze Lambert | project_manager | rank=4.0
480 | Catheryn Lambert | coordinator | rank=4.0
484 | Drina Lambert | engineer | rank=4.0
507 | Mynor Lambert | researcher | rank=3.5
533 | Alek Lambert | teacher | rank=2.5
560 | Ema Lambert | support_staff | rank=3.5
589 | Araia Lambert | engineer | rank=4.0
617 | Matteo Lambert | engineer | rank=3.0
637 | Clinton Lambert | data_analyst | rank=2.0

## The workload

The workload of a node (`node.workload`) is not entered: it is calculated as the sum of the weights of the incoming edges (the work sent to it). Work flows in both directions (a member sends courses to contributors, contributors send back assignments to be graded), which makes the graph rich. `graph.workload(id)` performs this calculation.

In [26]:
# Les 10 entités les plus chargées (somme du travail reçu).
ranked = sorted(master.nodes.values(), key=lambda n: -n.workload)
print(f"{'Label':30} | {'Role':24} | Load")
print('-' * 64)
for n in ranked[:10]:
    print(f'{n.label:30.30} | {n.role_family:24.24} | {n.workload}')

Label                          | Role                     | Load
----------------------------------------------------------------
ESPCI - Student Projects Team  |                          | 23.05
International Affairs - Data S |                          | 22.89
ESPCI - Research Coordination  |                          | 22.12
ENS - Data Science Team        |                          | 21.1
Observatoire de Paris - Studen |                          | 20.6
Observatoire de Paris - Resear |                          | 20.53
Mines Paris - Decision Support |                          | 19.6
Chimie ParisTech - Research Co |                          | 19.2
Dauphine - Data Science Team   |                          | 18.89
Mines Paris - Graph Analytics  |                          | 18.3


## The financial flow

Each person carries a signed financial flow, viewed from the organization's perspective: negative = the organization pays, positive = money flows in. Structural nodes do not have their own flow; their balance is the sum of the people who depend on them.

In [27]:
from orgalego.analysis.diagnosis import run_finance_report

run_finance_report(master)

MONEY FLOW  | sign: negative = paid out, positive = income
  Income (positive flows):            0
  Cost (negative flows):    -2,095,900
  NET balance:              -2,095,900

  Net balance per unit:
    Observatoire de Paris        -266,000
    Mines Paris                  -265,700
    ESPCI                        -263,100
    PSAI                         -259,900
    International Affairs        -259,600
    Chimie ParisTech             -258,900
    Dauphine                     -257,300
    ENS                          -253,000
    Housing                       -11,200
    Students                       -1,200

  Net balance per team:
    International Affairs - Graph Analytics       -58,800
    ESPCI - Data Science Team                     -58,300
    Chimie ParisTech - Graph Analytics Team       -58,000
    Dauphine - Graph Analytics Team               -57,900
    Mines Paris - Graph Analytics Team            -56,900
    Observatoire de Paris - Graph Analytics       -56,700
    O

# Part 2 - Algorithms and diagnostics

In [28]:
# Parameters - use search_node above to find ids.
source = "452"   # Hana Eustache (a student delegate)
target = "1"    # PSL University (the organization)
print('source:', source, '->', master.nodes[source].label)
print('target:', target, '->', master.nodes[target].label)

source: 452 -> Kammie Martin
target: 1 -> PSL University


### BFS - communication distance (number of hops)

In [29]:
from orgalego.analysis.diagnosis import run_bfs_diagnosis
run_bfs_diagnosis(master, source=source, target=target)

BFS  | source=452  target=1
  452 (Kammie Martin)  ->  1 (PSL University)
  Distance: 6 hop(s)
  Path: Kammie Martin -> Observatoire de Paris - Student Projects Team -> ESPCI - Student Projects Team -> Parvati Dupont -> Dauphine - Data Science Team -> Dauphine -> PSL University
  Interpretation: PSL University is 6 relation(s) away from Kammie Martin.


### Louvain - community / silo detection 

Unlike the others, Louvain is library-based (NetworkX);

In [30]:
from orgalego.analysis.diagnosis import run_louvain_diagnosis
run_louvain_diagnosis(master)

Louvain  | community detection (NetworkX)
  Modularity: 0.891
  36 communities:
    Community 1 (20): ['PSL University', 'Theo Engel', 'Student Council', 'Hana Eustache', 'PSAI', 'Dauphine', 'PSL Central Administration', 'Mines Paris', 'Ethics Committee', 'ENS', 'Observatoire de Paris', 'Academic Senate', 'Chimie ParisTech', 'ESPCI', 'Research Council', 'International Affairs', 'Housing Office', 'Maja Weiss', 'Igor Moreau', 'Sara Novak']
    Community 2 (17): ['Dimitra Rousseau', 'Blayklee Lambert', 'Kalinda Dupont', 'Aubrionna Bernard', 'Axel Fournier', 'Hanief Dubois', 'Saveyah Girard', 'Emmalou Bernard', 'Peony Dupont', 'PSAI - Data Science Team', 'Yazlin Roux', 'Bigyan Roux', 'Suzanne Eustache', 'Annaleigha Lambert', 'Evelyne Dubois', 'Comfort Moreau', 'Eljay Bernard']
    Community 3 (34): ['Flower Martin', 'Jalah Engel', 'Rayssa Girard', 'Alera Petit', 'Evanie Fournier', 'Attley Lopez', 'Kyale Simon', 'Adyn Girard', 'Averi Bernard', 'Meryn Dupont', 'Laila Rousseau', 'Kypton Schne

### Kruskal - minimal coordination skeleton (MST)

In [31]:
from orgalego.analysis.diagnosis import run_kruskal_diagnosis
run_kruskal_diagnosis(master)

Kruskal | original edges: 1071 | kept edges: 699 | removed edges: 372 | total weight: 1430.6

MST tree rooted at: PSL University

PSL University
├── Academic Senate (weight=3.5, rank=8.5)
├── Ethics Committee (weight=3.5, rank=8.5)
├── Research Council (weight=3.5, rank=8.5)
├── Observatoire de Paris (weight=2.25, rank=8.0)
│   ├── Dauphine (weight=1.87, rank=8.0)
│   │   ├── Dauphine - Research Coordination Team (weight=1.92, rank=6.0)
│   │   │   ├── Catalena Novak (weight=1.21, rank=4.0)
│   │   │   ├── Zorie Laurent (weight=3.97, rank=3.5)
│   │   │   ├── Faraja Novak (weight=2.88, rank=3.0)
│   │   │   ├── Zinnia Girard (weight=2.95, rank=3.0)
│   │   │   ├── Aqib Moreau (weight=1.22, rank=2.5)
│   │   │   │   ├── Betel Engel (weight=1.32, rank=3.5)
│   │   │   │   │   └── Kohlson Robert (weight=0.75, rank=2.5)
│   │   │   │   │       └── Mines Paris - Decision Support Team (weight=1.09, rank=6.0)
│   │   │   │   │           ├── ENS - Research Coordination Team (weight=2.18, rank=

### Dijkstra - least costly weighted path

In [32]:
from orgalego.analysis.diagnosis import run_dijkstra_diagnosis
run_dijkstra_diagnosis(master, source=source, target=target)

Dijkstra  | source=452  target=1
  Total cost: 10.34
  Path:
    452 -> Kammie Martin (support_staff)
    449 -> Deon Lefevre (student)
    42 -> Observatoire de Paris - Student Projects Team ()
    37 -> Observatoire de Paris ()
    1 -> PSL University ()
  Interpretation: communication must go through Deon Lefevre, Observatoire de Paris - Student Projects Team, Observatoire de Paris.


### Kosaraju - strongly connected components (directed graph)

In [33]:
from orgalego.analysis.diagnosis import run_kosaraju_diagnosis
run_kosaraju_diagnosis(master)

Kosaraju  | 261 components, 13 loop(s)
  Loop: ['PSL University', 'ENS', 'ENS - Data Science Team', 'Masen Novak', 'Emmalouise Moreau', 'Xylon Mercier', 'Portland Lefevre', 'Lolly Lopez', 'Eriana Girard', 'Brinna Lopez', 'Yazir Weiss', 'Raul Roux', 'Sharvin Lambert', 'Mines Paris - Graph Analytics Team', 'Timofey Novak', 'Chimie ParisTech - Research Coordination Team', 'Seraphine Schneider', 'International Affairs - Student Projects Team', 'Malori Fournier', 'Jaymien Mercier', 'Bowden Girard', 'Zarius Lefevre', 'Ahria Novak', 'Salsabil Girard', 'Rahi Moreau', 'Kallahan Simon', 'Naamah Novak', 'International Affairs', 'ESPCI', 'ESPCI - Graph Analytics Team', 'Indie Mercier', 'Demetrice Moreau', 'Fabyan Simon', 'Shukri Lopez', 'Bellen Durand', 'Kairah Mercier', 'Humphrey Robert', 'Tristyn Rousseau', 'Brentin Mercier', 'Graisen Garcia', 'Observatoire de Paris', 'Observatoire de Paris - Student Projects Team', 'Malala Petit', 'Trequan Laurent', 'Zarai Petit', 'Lyrix Mercier', 'Surya Simon'

### Betweenness - most central nodes
Unlike the others, Betweenness is library-based (NetworkX);

In [34]:
from orgalego.analysis.diagnosis import run_betweenness_diagnosis
run_betweenness_diagnosis(master, top_k=10)

Betweenness  | top 10 most central
   ID | Label                    | Role / type              | score
  ----------------------------------------------------------------
   49 | ESPCI                    | division                 | 0.2965
   55 | International Affairs    | division                 | 0.24
   37 | Observatoire de Paris    | division                 | 0.1681
   31 | ENS                      | division                 | 0.1651
   51 | ESPCI - Decision Support | team                     | 0.1508
   57 | International Affairs -  | team                     | 0.1316
   54 | ESPCI - Student Projects | team                     | 0.1265
   43 | Chimie ParisTech         | division                 | 0.1173
   56 | International Affairs -  | team                     | 0.099
   25 | Mines Paris              | division                 | 0.097
  Interpretation: high score = frequent transit point, hence a potential bottleneck if unavailable.
